<a href="https://colab.research.google.com/github/TonyQ2k3/pytorch-training/blob/main/nlp/pytorch_nlp.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install mlflow

In [2]:
import mlflow
import mlflow.pyfunc
import mlflow.spark
import os

from pyspark.sql import SparkSession, Row
from pyspark.ml import PipelineModel

In [ ]:
import pandas as pd

data = pd.read_csv('/content/Google Pixel_2025-05-15_00-16-54.csv')
data.rename(columns={'text': 'Text'}, inplace=True)
data['Label'] = 1.0

In [ ]:
data.head()

,product,Text,author,score,created,Label
0,Google Pixel,Pixel 9 Pro reportedly costs Google around 400...,a_Ninja_b0y,1873,2024-11-05,1.0
1,Google Pixel,"Sounds like a high price, tbh, I mean that’s o...",air_twee,702,2024-11-05,1.0
2,Google Pixel,"If I’m not mistaken, that’s in the same ballpa...",kiwipo17,122,2024-11-05,1.0
3,Google Pixel,Remember when Pixels were budget flagship phones?,1stltwill,57,2024-11-05,1.0
4,Google Pixel,So 999$ in retail is not that much If you add ...,Azuras33,225,2024-11-05,1.0


In [ ]:
def load_model_from_mlflow(model_uri):
    """
    Load a model from MLflow.
    :param model_uri: URI of the model in MLflow.
    :return: Loaded model.
    """
    # Load the model
    model = mlflow.spark.load_model(model_uri)
    return model

In [ ]:
mlflow.set_tracking_uri("https://dagshub.com/TranChucThien/kltn-sentiment-monitoring-mlops.mlflow")
spark = SparkSession.builder \
.appName("Load CountVectorizer_Model from MLflow") \
.getOrCreate()

# Load the model from MLflow
model_uri = "models:/CountVectorizer_Model/83"
model = load_model_from_mlflow(model_uri)

2025/05/21 15:42:30 INFO mlflow.spark: URI 'models:/CountVectorizer_Model/83/sparkml' does not point to the current DFS.
2025/05/21 15:42:30 INFO mlflow.spark: File 'models:/CountVectorizer_Model/83/sparkml' not found on DFS. Will attempt to upload the file.


In [ ]:
# Tạo DataFrame
df = spark.createDataFrame(data)

# Chạy transform qua model đã load
predictions = model.transform(df)

# Hiển thị kết quả
# predictions.show(truncate=False)
predictions.select("Text", "prediction").show(truncate=False)

+-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+----------+
|Text                                                                                                                                                                                                                                                                                                                                                                                                                                                                          

In [ ]:
from pyspark.sql.functions import udf
from pyspark.sql.functions import col

In [ ]:
def map_class(value):
  class_index_mapping = { 0: "Negative", 1: "Positive", 2: "Neutral" }
  return class_index_mapping[int(value)]

DataFrame[product: string, Text: string, author: string, score: bigint, created: string, Label: double, words: array<string>, filtered_words: array<string>, features: vector, rawPrediction: vector, probability: vector, prediction: double, class: string]

In [ ]:
predictions = predictions.withColumn("class", udf(map_class)(col("prediction")))

In [ ]:
predictions.select("class", "Text").show(truncate=False)

+--------+-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|class   |Text                                                                                                                                                                                                                                                                                                                                                                                                                                                                   